### General Functions

In [35]:
import json
import pandas as pd
import numpy as np
import torch
import os
from mxbai_rerank import MxbaiRerankV2
from google import genai
from google.genai.types import GenerateContentConfig
from langchain.prompts import PromptTemplate
from pydantic import RootModel, ValidationError
from typing import List
from dotenv import load_dotenv
import time

In [36]:
load_dotenv(dotenv_path="/home/ubuntu/LLM Rerankers/api_keys.env")

class RerankedIndices(RootModel[List[int]]):
    pass

In [37]:
print(torch.cuda.is_available())  # Should return True
print(torch.version.cuda)         # Should return 12.1
print(torch.backends.cudnn.version())  # Should print cuDNN version

True
12.1
90100


In [38]:
def check_gpu():
    """Verify if a GPU is present and return device type."""
    print("GPU load successful.")
    return "cuda" if torch.cuda.is_available() else "cpu"

In [39]:
### Load Texts
def load_json(file_path):
    """Load JSON data from file."""
    with open(file_path, 'r', encoding='utf-8') as f:
        print(f"JSON file loaded from {file_path}")
        return json.load(f)

### 20 Documents

In [40]:
def apply_rerank(reranked_vdb_20_documents_entry, retrieved_vdb_20_citations_entry, number_of_docs = 20):
    """Apply reranking to the retrieved citations."""

    # Set up evaluation per query
    rerank_order = reranked_vdb_20_documents_entry
    original_citations = retrieved_vdb_20_citations_entry
    
    # Case 1: Rerank order is None (or empty)
    if rerank_order is None:
        print("Case 1")
        reranked_citations = original_citations  # Keep the original order

    # Case 2: Rerank order is longer than the number of documents
    elif len(rerank_order) > number_of_docs:
        print("Case 2")
        rerank_order = [index for index in rerank_order if index < number_of_docs] # Remove excess indices

        reranked_citations = [original_citations[i] for i in rerank_order] # Rerank the order of the original citations according to the reranked indices
    
    # Case 3: Rerank order contains the indices with value greather than the number_of_docs
    elif any(index >= number_of_docs for index in rerank_order):
        print("Case 3")

        unexpected_indices = []
        unexpected_values = []

        for i in range(len(rerank_order)): # Iterate through the entire list to find unexpected indices
            if rerank_order[i] >= number_of_docs:
                unexpected_indices.append(i)
                unexpected_values.append(rerank_order[i])

        set_order = set(rerank_order)
        range_of_docs = set(range(number_of_docs))
        missing_indices = list(range_of_docs - set_order)

        # Sort both lists of unexpected values and missing values
        sorted_unexpected_values = sorted(unexpected_values)
        sorted_missing_indices = sorted(missing_indices) # Ensure missing_indices is sorted

        # Create a mapping based on the sorted order
        mapping = {}
        for i in range(len(sorted_unexpected_values)):
            if i < len(sorted_missing_indices):
                mapping[sorted_unexpected_values[i]] = sorted_missing_indices[i]
            else:
                print("Warning: Not enough missing values to map all unexpected values.")
                break


        # Apply the mapping to change the unexpected_values with the list_3 values at the unexpected_indices
        mapped_rerank_order = list(rerank_order) # Create a copy to modify

        for i in range(len(unexpected_indices)):
            index_to_change = unexpected_indices[i]
            original_value = mapped_rerank_order[index_to_change]
            if original_value in mapping:
                mapped_rerank_order[index_to_change] = mapping[original_value]
            else:
                print(f"Warning: No mapping found for value {original_value} at index {index_to_change}.")

        reranked_citations = [original_citations[i] for i in mapped_rerank_order] # Rerank the order of the original citations according to the reranked indices
        
    # All good
    else:
        print("All good")
        reranked_citations = [original_citations[i] for i in rerank_order] # Rerank the order of the original citations according to the reranked indices
    
    return reranked_citations

In [41]:
def add_common_entries(reranked_citations, google_citations, google_titles):
    """Add common entries and common titles from reranked citations to the final list."""
    common_citations = []
    common_titles = []

    for i in range(len(google_citations)):
        citation = google_citations[i]
        title = google_titles[i]

        if citation in reranked_citations:
            common_citations.append(citation)
            common_titles.append(title)
            
    return common_citations, common_titles

In [42]:
def fill_prompt(query, common_titles):

    # Define template for the prompt
    q_template = (
        "Given this list of retrieved titles (total: {number_of_titles}):\n"
        "{retrieved_titles}\n\n"

        "And given the query:\n"
        "{query}\n\n"

        "Rerank the titles above according to their relevance to the query. Rank them from most relevant to least relevant.\n"
        "After reranking, output the current order of their indices based on the original order. Format it as a Python list of integers. For example, [9, 7, 6, 5, 3, 1, 8, 2, 4, 0].\n"
        "There is no need to explain your answers. You can simply output the Python list of indices.\n\n"

        "Make sure that:\n"
        "- The list contains only integers between 0 and {number_of_titles_minus_one}.\n"
        "- The length of the list is exactly {number_of_titles}."
    )

    q_prompt = PromptTemplate(
        input_variables=["number_of_titles", "retrieved_titles", "query", "number_of_titles_minus_one"],
        template=q_template
    )

    number_of_titles = len(common_titles)
    number_of_titles_minus_one = number_of_titles - 1

    filled_prompt = q_prompt.format(
                number_of_titles=number_of_titles,
                retrieved_titles=common_titles,
                query=query,
                number_of_titles_minus_one=number_of_titles_minus_one
            )
    
    return filled_prompt

In [43]:
def generate(filled_prompt):

    # Initialize the Google Gemini client
    client = genai.Client(
        api_key=os.getenv("GEMINI_API_KEY"),
    )

    model = "gemini-2.0-flash-thinking-exp-01-21"
    contents = filled_prompt
    
    generate_content_config = GenerateContentConfig(
        temperature=0.1,
        max_output_tokens=15000,
        response_mime_type="text/plain",
    )

    max_retries = 5
    initial_delay = 5
    
    for attempt in range(max_retries + 1):
        try:
            full_response = client.models.generate_content(
                model=model,
                contents=contents,
                config=generate_content_config,
            )

            # Default response
            text_response = "[]"
            structured_response = None  # Initialize structured_response

            # Check if the response is valid
            if full_response and full_response.candidates and full_response.candidates[0]:
                # Extracts the candidate response
                candidate = full_response.candidates[0]

                if candidate.content and candidate.content.parts:
                    # Extract text response
                    text_response = ""
                    for each in candidate.content.parts:
                        text_response += each.text

                    # Clean the text response: remove code block markers and strip whitespace
                    cleaned_response = text_response.strip()
                    if cleaned_response.startswith("```") and cleaned_response.endswith("```"):
                        # Remove the outer ``` if present
                        cleaned_response = cleaned_response[3:-3].strip()
                        # Remove language identifier if present (e.g., ```json)
                        if cleaned_response.startswith("json"):
                            cleaned_response = cleaned_response[4:].strip()
                        if cleaned_response.startswith("python"):
                            cleaned_response = cleaned_response[6:].strip()

                    # Attempt to parse the cleaned text response as JSON and validate with Pydantic
                    try:
                        json_output = json.loads(cleaned_response)
                        structured_response = RerankedIndices.model_validate_json(json.dumps(json_output))
                    except (json.JSONDecodeError, ValidationError) as e:
                        print(f"Error parsing or validating JSON: {e}")
                        print(f"Failed to parse JSON. Raw response: '{text_response}'")
                        text_response = "[]"
                        structured_response = None

            return text_response, structured_response

        except genai.errors.ServerError as e:
            if e.code == 503:
                print(f"Model overloaded (attempt {attempt + 1}/{max_retries + 1}). Retrying in {initial_delay * (2 ** attempt)} seconds...")
                if attempt < max_retries:
                    time.sleep(initial_delay * (2 ** attempt))  # Exponential backoff
                else:
                    print("Max retries reached. Skipping this query.")
                    return "[]", None  # Return default values after max retries
            else:
                print(f"An unexpected ServerError occurred: {e}")
                return "[]", None
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            return "[]", None

In [44]:
def rerank_common_entries(query, common_entries, common_titles):

    adjusted_common_entries = []

    # Generate the prompt
    filled_prompt = fill_prompt(query, common_titles)

    # Rerank the entries based on the titles
    text_response, structured_response = generate(filled_prompt)
    
    print(structured_response)

    for i in structured_response.root:
        adjusted_common_entries.append(common_entries[i])
    
    return adjusted_common_entries


In [45]:
def ensure_unique_citations_and_titles(list_of_citations, list_of_titles, common_citations, common_titles, top_k):
    """Ensure that the list of citations and the list of titles is unique."""
    unique_citations = []
    unique_titles = []

    for citation in list_of_citations:

        if citation not in common_citations and citation not in unique_citations:
            unique_citations.append(citation) # Track this citation
            
        if len(unique_citations) == top_k:  # Stop once we have enough unique results
            break
    
    for title in list_of_titles:

        if title not in common_titles and title not in unique_titles:
            unique_titles.append(title) # Track this title

        if len(unique_titles) == top_k: # Stop once we have enough unique results
            break

    return unique_citations, unique_titles

In [46]:
def populate_final_citations_h1(final_list, unique_vector_db_citations, unique_google_citations, top_k):
    """Populate the final list of citations using Google citations (and vector database as fallback)."""

    needed = top_k - len(final_list)

    google_index = 0
    vdb_index = 0

    for _ in range(needed):
        if google_index < len(unique_google_citations):
            final_list.append(unique_google_citations[google_index])
            google_index += 1
        elif vdb_index < len(unique_vector_db_citations):
            final_list.append(unique_vector_db_citations[vdb_index])
            vdb_index += 1
        else:
            break  # both sources exhausted
    
    return final_list

In [47]:
def populate_final_citations_h2(final_list, unique_vector_db_citations, unique_google_citations, top_k):
    """Populate the final list by alternating between Google and vector DB citations, prioritizing Google for the final slot."""

    needed = top_k - len(final_list)

    google_index = 0
    vdb_index = 0

    turn = 0  # 0 = Google, 1 = VDB

    for i in range(needed):
        remaining_slots = needed - i

        # Last slot: prioritize Google
        if remaining_slots == 1:
            if google_index < len(unique_google_citations):
                final_list.append(unique_google_citations[google_index])
                google_index += 1
            elif vdb_index < len(unique_vector_db_citations):
                final_list.append(unique_vector_db_citations[vdb_index])
                vdb_index += 1
            break

        if turn == 0:
            if google_index < len(unique_google_citations):
                final_list.append(unique_google_citations[google_index])
                google_index += 1
            elif vdb_index < len(unique_vector_db_citations):
                final_list.append(unique_vector_db_citations[vdb_index])
                vdb_index += 1
            else:
                break
            turn = 1  # switch to VDB
        else:
            if vdb_index < len(unique_vector_db_citations):
                final_list.append(unique_vector_db_citations[vdb_index])
                vdb_index += 1
            elif google_index < len(unique_google_citations):
                final_list.append(unique_google_citations[google_index])
                google_index += 1
            else:
                break
            turn = 0  # switch to Google

    return final_list

In [48]:
def eval_citations(target_citation, final_list_of_citations, recall_scores, mrr_scores, ndcg_scores):
        
    # Checking whether the target citation is in the unique citations
    citation_set = {target_citation}  # Use set for O(1) lookup

    found_match = False

    for index, citation in enumerate(final_list_of_citations):

        if citation.strip() in citation_set:
            recall_scores.append(1)
            mrr_scores.append(1 / (index + 1))
            ndcg_scores.append(1 / np.log2(index + 2))
            found_match = True
            break  # Exit the inner loop once a match is found

    if not found_match:
        recall_scores.append(0)
        mrr_scores.append(0)
        ndcg_scores.append(0)

In [49]:
# Define parameters
number_of_documents = 20
top_k = 10

reranked_vdb_20_documents_path = "/home/ubuntu/LLM Rerankers/Reranked Responses/Gemini Structured Responses/Gemini_structured-reranked-20-indices.json"
retrieved_vdb_20_citations_path =  "/home/ubuntu/RAG Pipeline/March 27 Citations/1024-256-20-retrieved_citations.json"
retrieved_vdb_20_titles_path = "/home/ubuntu/RAG Pipeline/March 27 Titles/1024-256-20-retrieved_titles.json"

google_search_results_path = "/home/ubuntu/LLM and Search Experiments/Cleaned Text Responses/QTTTAtoC_cleaned-text-responses.json"

eval_csv_path = "/home/ubuntu/[March 27, 2025] Final Dataset/sampled_final_cleaned_acl_global_context_dataset_eval.csv"

# Load data
reranked_vdb_20_documents = load_json(reranked_vdb_20_documents_path)
retrieved_vdb_20_citations = load_json(retrieved_vdb_20_citations_path)
retrieved_vdb_20_titles = load_json(retrieved_vdb_20_titles_path)

google_search_results = load_json(google_search_results_path)
google_search_citations = google_search_results["citations"]
google_search_titles = google_search_results["titles"]

counter_for_more_than_one_common_entries = 0

eval_df = pd.read_csv(eval_csv_path)
citation_contexts = eval_df["masked_cit_context"].tolist()
citation_targets = eval_df['masked_token_target'].tolist()
print("Citation contexts loaded successfully.")
print("Citation targets loaded successfully.")

JSON file loaded from /home/ubuntu/LLM Rerankers/Reranked Responses/Gemini Structured Responses/Gemini_structured-reranked-20-indices.json
JSON file loaded from /home/ubuntu/RAG Pipeline/March 27 Citations/1024-256-20-retrieved_citations.json
JSON file loaded from /home/ubuntu/RAG Pipeline/March 27 Titles/1024-256-20-retrieved_titles.json
JSON file loaded from /home/ubuntu/LLM and Search Experiments/Cleaned Text Responses/QTTTAtoC_cleaned-text-responses.json
Citation contexts loaded successfully.
Citation targets loaded successfully.


In [50]:
recall_scores = []
mrr_scores = []
ndcg_scores = []

for i in range(len(citation_contexts)):

    reranked_vdb_20_documents_entry = reranked_vdb_20_documents[i]
    retrieved_vdb_20_citations_entry = retrieved_vdb_20_citations[i]

    # Apply the reranking
    reranked_vdb_20_citations = apply_rerank(reranked_vdb_20_documents_entry, retrieved_vdb_20_citations_entry)
    retrieved_vdb_20_titles_entry = retrieved_vdb_20_titles[i]

    citations_from_google = google_search_citations[i]
    titles_from_google = google_search_titles[i]

    query = citation_contexts[i]
    target_citation = citation_targets[i]

    # Add common entries to the final list
    common_citations, common_titles = add_common_entries(reranked_vdb_20_citations, citations_from_google, titles_from_google)

    # Rerank the common entries based on their paper title (by relevance to the query)
    if len(common_citations) > 1:
        counter_for_more_than_one_common_entries += 1
        rr_temp_final_citations = rerank_common_entries(query, common_citations, common_titles)
        time.sleep(6)
    else:
        rr_temp_final_citations = common_citations # No reranking done

    # Keep only the highest-ranked document for each unique citation in the vdb that are not in the common entires
    vdb_20_unique_citation_items, vdb_20_unique_titles = ensure_unique_citations_and_titles(reranked_vdb_20_citations, retrieved_vdb_20_titles_entry, rr_temp_final_citations, common_titles, top_k)

    # Keep only the highest-ranked document for each unique citation in the google search that are not in the common entries
    unique_citations_from_google, unique_titles_from_google = ensure_unique_citations_and_titles(citations_from_google, titles_from_google, rr_temp_final_citations, common_titles, top_k)
    
    # Complete the final list of citations
    final_list_of_citations = populate_final_citations_h2(rr_temp_final_citations, vdb_20_unique_citation_items, unique_citations_from_google, top_k)
    
    # Compare masked citation with the final list of citations
    eval_citations(target_citation, final_list_of_citations, recall_scores, mrr_scores, ndcg_scores)

    if (i + 1) % 15 == 0:

            # Compute and print running metrics
            running_avg_recall = np.mean(recall_scores)
            running_avg_mrr = np.mean(mrr_scores)
            running_avg_ndcg = np.mean(ndcg_scores)
            
            print(f"Processed {i+1} queries...")
            print(f"Running Metrics: Average Recall: {running_avg_recall:.4f} | Average MRR: {running_avg_mrr:.4f} | Average NDCG: {running_avg_ndcg:.4f}")

# Aggregate metrics over all queries
count = len(recall_scores)
overall_metrics = {
    "Recall": np.mean(recall_scores) if count > 0 else 0,
    "MRR": np.mean(mrr_scores) if count > 0 else 0,
    "NDCG": np.mean(ndcg_scores) if count > 0 else 0,
    "count": count
}
    
print("Overall Metrics:")
print(overall_metrics)
print(f"Counter for more than one common entires: {counter_for_more_than_one_common_entries}")

All good
All good
All good
All good
All good
All good
All good
All good
All good
All good
All good
All good
All good
All good
All good


root=[1, 0]
Processed 15 queries...
Running Metrics: Average Recall: 0.9333 | Average MRR: 0.8429 | Average NDCG: 0.8643
All good
All good
All good
root=[0, 1]
All good
All good
All good
All good
All good
All good
All good
root=[1, 0, 2]
All good
All good
All good
All good
All good
Processed 30 queries...
Running Metrics: Average Recall: 0.8667 | Average MRR: 0.7159 | Average NDCG: 0.7527
All good
All good
All good
All good
root=[0, 1]
All good
All good
All good
All good
root=[1, 0]
All good
All good
All good
All good
root=[1, 0]
All good
All good
All good
Processed 45 queries...
Running Metrics: Average Recall: 0.8000 | Average MRR: 0.6272 | Average NDCG: 0.6704
All good
All good
All good
root=[1, 0, 2]
All good
All good
All good
All good
root=[3, 1, 0, 2]
All good
root=[0, 1]
All good
All good
All good
All good
root=[1, 0]
All good
All good
All good
Processed 60 queries...
Running Metrics: Average Recall: 0.8333 | Average MRR: 0.6315 | Average NDCG: 0.6826
All good
root=[0, 1]
All go

### Experiments

In [25]:
retrieved_vdb_50_documents_path = f"/home/ubuntu/RAG Pipeline/March 27 Chunks/1024-256-50-retrieved_chunks.json"
retrieved_vdb_50_citations_path =  f"/home/ubuntu/RAG Pipeline/March 27 Citations/1024-256-50-retrieved_citations.json"

# Load data
retrieved_vdb_50_documents = load_json(retrieved_vdb_50_documents_path)
retrieved_vdb_50_citations = load_json(retrieved_vdb_50_citations_path)

JSON file loaded from /home/ubuntu/RAG Pipeline/March 27 Chunks/1024-256-50-retrieved_chunks.json
JSON file loaded from /home/ubuntu/RAG Pipeline/March 27 Citations/1024-256-50-retrieved_citations.json


In [26]:
reranker_model_name = "mixedbread-ai/mxbai-rerank-base-v2" # change to mxbai-rerank-large-v2 if you want
reranker_model = MxbaiRerankV2(reranker_model_name, device=check_gpu())

GPU load successful.


In [27]:
def apply_rerank_mxbai(query, retrieved_vdb_documents_entry, retrieved_vdb_citations_entry, reranker_model):
    """Apply reranking to the retrieved citations."""

    reranked_citations = []

    try:
        with torch.no_grad():
            results = reranker_model.rank(query, retrieved_vdb_documents_entry)

        # Sort results by score in descending order
        results.sort(key=lambda x: x.score, reverse=True)
        sorted_indices = [result.index for result in results]

    except torch.cuda.OutOfMemoryError:
        print("OOM during reranking. Returning original chunks.")
        sorted_indices = [i for i in range(len(retrieved_vdb_documents_entry))]

    # Sort the original results based on the sorted indices
    for i in sorted_indices:
        reranked_citations.append(retrieved_vdb_citations_entry[i])

    return reranked_citations

In [ ]:
sample = apply_rerank_mxbai(citation_contexts[0], retrieved_vdb_50_documents[0], retrieved_vdb_50_citations[0], reranker_model)

print(f"Original citations: {retrieved_vdb_50_citations[0]}")
print(f"Reranked citations: {sample}")

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


/opt/conda/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2708: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Original citations: ['Blunsom and Cohn, 2006', 'Blunsom and Cohn, 2006', 'Rauf and Schwenk, 2009', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Blunsom et al., 2009', 'Le et al., 2012', 'Wuebker et al., 2010', 'Foster et al., 2006', 'Chiang, 2005', 'Chiang, 2005', 'Chiang, 2005', 'Chiang, 2005', 'Chiang, 2005', 'Chiang, 2005', 'Chiang, 2005', 'Chiang, 2005', 'Chiang, 2005', 'Chiang, 2005', 'Chiang, 2005', 'Marcu and Wong, 2002', 'Dahlmeier and Ng, 2011', 'Dahlmeier and Ng, 2011', 'MacCartney et al., 2008', 'Liu et al., 2006', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn et al., 2003', 'Koehn

: 